In [1]:
from colorama import Fore 
from docling.document_converter import DocumentConverter
from docling.datamodel.base_models import InputFormat
from docling.chunking import HybridChunker
from transformers import AutoTokenizer
from pprint import pprint
from transformers import AutoTokenizer

/home/mrosaria/Projects/NLP/GymRat/ratenv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
import unicodedata

In [3]:
from pathlib import Path
import sys
sys.path.append(str(Path('../python').resolve()))
from create_clean_chunks import CleanText

# Load and Chunk Document

In [4]:
file="../data/books/rebuilding_milo.pdf"
converter = DocumentConverter()
result = converter.convert(file) 

In [ ]:
#result.document.texts[176]

SectionHeaderItem(self_ref='#/texts/176', parent=RefItem(cref='#/body'), children=[], content_layer=<ContentLayer.BODY: 'body'>, label=<DocItemLabel.SECTION_HEADER: 'section_header'>, prov=[ProvenanceItem(page_no=28, bbox=BoundingBox(l=76.937, t=207.27600000000007, r=190.946, b=183.76700000000005, coord_origin=<CoordOrigin.BOTTOMLEFT: 'BOTTOMLEFT'>), charspan=(0, 9))], orig='The Spine', text='The Spine', formatting=None, hyperlink=None, level=1)

In [36]:
for i, item in enumerate(result.document.texts[160:200]):
    visible = item.text.replace("\n", "\\n").replace("\t", "\\t")
    #pprint(f"--- Text {i} ---\n{visible}")
    print(f"--- Text {i} ---")
    print(repr(item.orig))  # shows \n, \t, etc.
    print(repr(item.text))

--- Text 0 ---
'· The MRI shows only anatomy, not function.'
'The MRI shows only anatomy, not function.'
--- Text 1 ---
"What if I told you that disc bulges are fairly common and often show up on lumbar spine MRI scans? It's true! Many people are walking around with disc bulges but have zero back pain."
"What if I told you that disc bulges are fairly common and often show up on lumbar spine MRI scans? It's true! Many people are walking around with disc bulges but have zero back pain."
--- Text 2 ---
'Research has estimated that almost a third of healthy, pain-free 20year-olds have a disc bulge in their spine.  6  This number increases by 10 percent for every decade of life, meaning that half of all 40-year-olds likely have a disc bulge yet are not experiencing back pain.'
'Research has estimated that almost a third of healthy, pain-free 20year-olds have a disc bulge in their spine.  6  This number increases by 10 percent for every decade of life, meaning that half of all 40-year-olds l

In [6]:
class HeaderPreservingChunker:
    """
    Wraps HybridChunker but ensures short lines (headers, captions, etc.)
    are merged into the following content chunk.
    """

    def __init__(self, tokenizer, max_tokens=512, min_tokens=50, short_threshold=40):
        self.base_chunker = HybridChunker(
            tokenizer=tokenizer,
            max_tokens=max_tokens,
            min_tokens=min_tokens,
            respect_sentence_boundaries=True,
            merge_peers=True,
            merge_tiny=True
        )
        self.short_threshold = short_threshold

    def chunk(self, dl_doc):
        """
        Chunk the doc and merge orphan short lines into the next chunk.
        """
        raw_chunks = list(self.base_chunker.chunk(dl_doc=dl_doc))

        merged_chunks = []
        buffer = None

        for chunk in raw_chunks:
            text = chunk.text.strip()

            # If chunk is very short (potential header), buffer it
            if len(text) <= self.short_threshold and "\n" not in text:
                if buffer is None:
                    buffer = text
                else:
                    buffer += " " + text
                continue

            # If we have a buffered header, prepend it to this chunk
            if buffer:
                new_text = buffer + "\n" + text
                chunk.text = new_text
                buffer = None

            merged_chunks.append(chunk)

        # If only orphan headers remain at the end, merge them into the last chunk
        if buffer and merged_chunks:
            merged_chunks[-1].text = buffer + "\n" + merged_chunks[-1].text

        return merged_chunks

In [ ]:
model_id = "meta-llama/Llama-3.1-8B-Instruct" #TinyLlama/TinyLlama-1.1B-Chat-v1.0"


In [20]:

tokenizer = AutoTokenizer.from_pretrained(model_id)

# Use the custom chunker
chunker = HeaderPreservingChunker(tokenizer=tokenizer, max_tokens=128, min_tokens=50)
chunks = chunker.chunk(result.document)
# for i, c in enumerate(chunks):
#     print(f"\n--- Chunk {i} ---\n{c.text[:400]}...")
len(chunks)


1711

In [25]:
pprint(chunks[100].text)

('BODYWEIGHT SQUAT WITH BUTTWINK\n'
 'Now, if you are just performing a few bodyweight squats and butt winking '
 "occurs, it's likely not a big deal. Minimal power is generated at the\n"
 'spine during a normal-tempo air squat. However, as soon as you add a '
 'barbell, things change. If butt winking continues under load, the power\n'
 'generated at the spine increases at one or two specic joints of the lumbar '
 'spine (usually L4/5 and L5/S1). Therefore, when you have a stress')


In [5]:
import unicodedata

def clean_word(word: str) -> str:
    """Replace bad glyphs or unusual non-ASCII characters in a word."""

    # Normalize (compatibility decomposition: splits ligatures/combining marks)
    #word = unicodedata.normalize("NFKC", word)

    replacements = {
        # Common ligatures
        "\uFB01": "fi",     # ﬁ
        "\uFB02": "fl",     # ﬂ
        "\uFB03": "ffi",    # ﬃ
        "\uFB04": "ffl",    # ﬄ
        "\uFB00": "ff",     # ﬀ
        "\uFB05": "ft",     # ﬅ
        "\uFB06": "st",     # ﬆ
        "\u0346": "fl",  
        "\u0345": "fi",
        "\u00EF": "i",

        # Accented/diacritic variants
        "\u00EF": "i",      # ï
        "\u00EE": "i",      # î
        "\u00ED": "i",      # í
        "\u00EC": "i",      # ì
        "\u00E1": "a",      # á
        "\u00E0": "a",      # à
        "\u00E2": "a",      # â
        "\u00E4": "a",      # ä
        "\u00E3": "a",      # ã
        "\u00E5": "a",      # å
        "\u00E7": "c",      # ç
        "\u00E9": "e",      # é
        "\u00E8": "e",      # è
        "\u00EA": "e",      # ê
        "\u00EB": "e",      # ë
        "\u00F1": "n",      # ñ
        "\u00F3": "o",      # ó
        "\u00F2": "o",      # ò
        "\u00F4": "o",      # ô
        "\u00F6": "o",      # ö
        "\u00F5": "o",      # õ
        "\u00FA": "u",      # ú
        "\u00F9": "u",      # ù
        "\u00FB": "u",      # û
        "\u00FC": "u",      # ü
        "\u00FD": "y",      # ý
        "\u00FF": "y",      # ÿ

        # Dashes and hyphens
        "\u2013": "-",      # –
        "\u2014": "-",      # —
        "\u2212": "-",      # −
        "\u2012": "-",      # ‒

        # Quotes
        "\u201C": "\"",     # “
        "\u201D": "\"",     # ”
        "\u201E": "\"",     # „
        "\u201F": "\"",     # ‟
        "\u2018": "'",      # ‘
        "\u2019": "'",      # ’
        "\u201A": "'",      # ‚
        "\u201B": "'",      # ‛

        # Ellipsis
        "\u2026": "...",    # …

        # Spaces
        "\u00A0": " ",      # non-breaking space
        "\u2002": " ",      # en space
        "\u2003": " ",      # em space
        "\u2009": " ",      # thin space
        "\u202F": " ",      # narrow no-break space
        "\u3000": " ",      # ideographic space
        "\u200B": "",       # zero-width space
        "\u200C": "",       # zero-width non-joiner
        "\u200D": "",       # zero-width joiner

        # Misc
        "\u2022": "-",      # • bullet
        "\u00B7": "-",      # · middle dot
        "\u2219": "-",      # ∙ dot operator
        "\u25CB": "o",      # ○
        "\u00B0": " degrees", # °
        "\u00BC": "1/4",    # ¼
        "\u00BD": "1/2",    # ½
        "\u00BE": "3/4",    # ¾
    }

    for bad, good in replacements.items():
        word_og = word  # Keep original for debugging
        word = word.replace(bad, good)
        if word != word_og:
            print(Fore.RED + f"Replaced '{bad}' with '{good}' in word: {word_og} -> {word}" + Fore.RESET)


    return word


# Clean Chunks

In [6]:
class HeaderPreservingChunker:
    """
    Wraps HybridChunker but ensures short lines (headers, captions, etc.)
    are merged into the following content chunk.
    """

    def __init__(self, file_path, tokenizer, max_tokens=512, min_tokens=50, short_threshold=40):
        self.file_path = file_path
        self.base_chunker = HybridChunker(
            tokenizer=tokenizer,
            max_tokens=max_tokens,
            min_tokens=min_tokens,
            respect_sentence_boundaries=True,
            merge_peers=True,
            merge_tiny=True
        )
        self.short_threshold = short_threshold

        self.chunks = self.create_chunks(self.load_document().document)
        #self.chunks = self.create_chunks(results)
        self.non_ascii_words, self.private_unicode_words, self.non_ascii_chars  = self.get_problematic_lists(self.chunks)

    def load_document(self):
        """
        Load the document from the specified file path.
        """
        converter = DocumentConverter()
        return converter.convert(self.file_path)

    def create_chunks(self, dl_doc):
        """
        Chunk the doc and merge orphan short lines into the next chunk.
        """
        raw_chunks = list(self.base_chunker.chunk(dl_doc=dl_doc))

        merged_chunks = []
        buffer = None

        for chunk in raw_chunks:
            text = chunk.text.strip()

            # If chunk is very short (potential header), buffer it
            if len(text) <= self.short_threshold and "\n" not in text:
                if buffer is None:
                    buffer = text
                else:
                    buffer += " " + text
                continue

            # If we have a buffered header, prepend it to this chunk
            if buffer:
                new_text = buffer + "\n" + text
                chunk.text = new_text
                buffer = None

            merged_chunks.append(chunk)

        # If only orphan headers remain at the end, merge them into the last chunk
        if buffer and merged_chunks:
            merged_chunks[-1].text = buffer + "\n" + merged_chunks[-1].text

        return merged_chunks

    def get_problematic_lists(self, chunks):
        # Create lists with problematic words or characters for all the chunks in the provided file
        # Used later for the CleanText class to remove or fix this words. Also used manually for debbuging and checking the quality of the resulting cleaned chunks 
        non_ascii_words, private_unicode_words, non_ascii_chars = set(), set(), set()
        for chunk in chunks:
            #Normalize text for consistent Unicode handling
            text = unicodedata.normalize("NFKC", chunk.text)

            clean_text = CleanText(text)

            non_ascii_words.update(clean_text.find_words_with_non_ascii())
            private_unicode_words.update(clean_text.find_words_with_private_unicode())
            non_ascii_chars.update(clean_text.extract_non_ascii_characters())

        non_ascii_words = list(sorted(non_ascii_words))
        private_unicode_words = list(sorted(private_unicode_words))
        non_ascii_chars = list(sorted(non_ascii_chars))

        #print(len(non_ascii_words), len(private_unicode_words), len(non_ascii_chars))
        return non_ascii_words, private_unicode_words, non_ascii_chars

    def get_clean_text(self, text):
        return CleanText(text, self.non_ascii_words)

    def clean_chunk(self, text):
        clean_text = self.get_clean_text(text)
        words = text.split() # chunk == text
        clean_chunk = ""
        for word in words:
            #new_text =  self.clean_word(word)
            new_text =  clean_word(word)
            if new_text == None:
                print(word, "    NONE \\n ")
            clean_chunk += new_text + " "
        return clean_chunk.strip()

    def get_clean_chunks(self):
        text_chunks=[]
        n=0
        #chunks = self.get_chunks()
        for chunk in self.chunks:
            text_chunk = chunk.text 
            print(f"Processing chuck number: {n}")
            text_chunk = self.clean_chunk(text_chunk)
            text_chunks.append(text_chunk)
            n+=1
        return text_chunks


In [7]:
file="../data/books/rebuilding_milo.pdf"
model_id = "meta-llama/Llama-3.1-8B-Instruct" #TinyLlama/TinyLlama-1.1B-Chat-v1.0"
tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token
# Use the custom chunker
cc = HeaderPreservingChunker(file_path=file, tokenizer=tokenizer,  max_tokens=128)
x = cc.get_clean_chunks()
print(x[:10])

Processing chuck number: 0
Processing chuck number: 1
Processing chuck number: 2
Processing chuck number: 3
Processing chuck number: 4
Processing chuck number: 5
Processing chuck number: 6
Processing chuck number: 7
Processing chuck number: 8
Processing chuck number: 9
Processing chuck number: 10
Processing chuck number: 11
Processing chuck number: 12
Processing chuck number: 13
Processing chuck number: 14
Processing chuck number: 15
Processing chuck number: 16
Processing chuck number: 17
Processing chuck number: 18
Processing chuck number: 19
Processing chuck number: 20
Processing chuck number: 21
Processing chuck number: 22
Processing chuck number: 23
Processing chuck number: 24
Processing chuck number: 25
Processing chuck number: 26
Processing chuck number: 27
Processing chuck number: 28
Processing chuck number: 29
Processing chuck number: 30
Processing chuck number: 31
Processing chuck number: 32
Processing chuck number: 33
Processing chuck number: 34
Processing chuck number: 35
Pr

In [15]:
pprint(x[34])

('Leading up to the competition, I decided to turn up the heat in my '
 'preparation. I started to pull two-a-day training sessions. I would wake up '
 'at 6 a.m. to get my squats and pulls in before heading to grad school '
 'classes for the day. I would return later to nish up whatever snatches, '
 'cleans, or jerks the program had in store. There was no way I was going to '
 'pass up my chance to perform at my very best on the national stage.')


In [14]:
# Example list
my_list = x

# Save to a file (one string per line)
with open("../data/rebuilding_milo_chunks_docling_max_tokens128_min_tokens50_meta_llama3p18B.txt", "w", encoding="utf-8") as f:
    for item in my_list:
        f.write(item + "\n")


In [ ]:
#x[35].split(' ')

['Does',
 'this',
 'mean',
 'every',
 'rounded-spine',
 'lift',
 'will',
 'create',
 'a',
 'bulging',
 'disc?',
 'Not',
 'necessarily.',
 'Many',
 'factors',
 'play',
 'into',
 'this',
 'discussion.',
 'Every',
 'athlete',
 'tolerates',
 'the',
 'forces',
 'of',
 'bending',
 'their',
 'spine',
 'differently.',
 'This',
 'is',
 'why',
 'elite',
 'gymnasts',
 'can',
 'bend',
 'themselves',
 'in',
 'half',
 'over',
 'and',
 'over,',
 'yet',
 'attempting',
 'the',
 'same',
 'movements',
 'would',
 'eventually',
 'spell',
 'disaster',
 'for',
 'an',
 'elite',
 'heavyweight',
 'powerlifter.',
 "Here's",
 'a',
 'great',
 'way',
 'to',
 'understand',
 'this',
 'principle.',
 'Some',
 'tree',
 'branches',
 'are',
 'slender',
 'and',
 'bend',
 'easily',
 'over',
 'and',
 'over',
 'again.',
 'Other',
 'branches',
 'are',
 'thicker',
 'and',
 'begin',
 'to',
 'snap',
 'in',
 'two',
 'after',
 'a',
 'few',
 'bends.',
 'Every',
 'body',
 'is',
 'different.',
 'Depending',
 'on',
 'a',
 'number',
 'o

In [103]:
chunks_text = [x.text for x in chunks]

In [104]:
chunks_text[35]

"Does this mean every rounded-spine lift will create a bulging disc? Not necessarily. Many factors play into this discussion. Every athlete tolerates\nthe forces of bending their spine differently. This is why elite gymnasts can bend themselves in half over and over, yet attempting the same\nmovements would eventually spell disaster for an elite heavyweight powerlifter. Here's a great way to understand this principle.\nSome tree branches are slender and bend easily over and over again.\nOther branches are thicker and begin to snap in two after a few bends.\nEvery body is different. Depending on a number of factors, such as your anatomy, genetics, the amount of weight lifted, and the degree of poor technique, your body may be more or less resilient to developing a disc bulge.  18\nThis doesn't mean you should fear exion of the spine. However, you must understand that the mechanism that creates a disc bulge includes exion. If the force applied to the spine as it exes is low, power genera

In [105]:
#for i, word in enumerate(x[35].split(' ')):
for i, word in enumerate(chunks_text[35].split(' ')):

    print(i, word, clean_word(word))

0 Does Does
1 this this
2 mean mean
3 every every
4 rounded-spine rounded-spine
5 lift lift
6 will will
7 create create
8 a a
9 bulging bulging
10 disc? disc?
11 Not Not
12 necessarily. necessarily.
13 Many Many
14 factors factors
15 play play
16 into into
17 this this
18 discussion. discussion.
19 Every Every
20 athlete athlete
21 tolerates
the tolerates
the
22 forces forces
23 of of
24 bending bending
25 their their
26 spine spine
27 differently. differently.
28 This This
29 is is
30 why why
31 elite elite
32 gymnasts gymnasts
33 can can
34 bend bend
35 themselves themselves
36 in in
37 half half
38 over over
39 and and
40 over, over,
41 yet yet
42 attempting attempting
43 the the
44 same
movements same
movements
45 would would
46 eventually eventually
47 spell spell
48 disaster disaster
49 for for
50 an an
51 elite elite
52 heavyweight heavyweight
53 powerlifter. powerlifter.
54 Here's Here's
55 a a
56 great great
57 way way
58 to to
59 understand understand
60 this this
61 principl

In [107]:
text = chunks_text[35].split(' ')[148]
for ch in text:
    print(f"'{ch}' -> U+{ord(ch):04X}")

'e' -> U+0065
'x' -> U+0078
'i' -> U+0069
'o' -> U+006F
'n' -> U+006E
'.' -> U+002E


In [76]:
cc.non_ascii_words[20:50]

['naïve',
 '©',
 '©Bruce',
 '©Jerry',
 '»',
 'Öhberg,',
 '×',
 'Ø.',
 'Łabuz-Roszak,',
 'Škarabot,',
 '•']

# Generate content

In [6]:
from create_qa import GenerateQAContent
from tqdm import tqdm

In [5]:
file_name = "rebuilding_milo_chunks_docling_max_tokens512_min_tokens50.txt"
   
# file_name = "../notebooks/docling_chunk_text_context_llm_tokenizer.json"#"llama_chunks_text_chuck_size250_overlap30.json"
gc = GenerateQAContent(file_name)
text = gc.get_text()
raw = gc.generate_content(n_max_chunks=3)
print(raw)

Device set to use cuda:0
  0%|          | 0/1 [00:00<?, ?it/s]

i: 0
Processing batch with samples (0, 16) 


100%|██████████| 1/1 [01:28<00:00, 88.63s/it]

['<|system|>\nYou are a concise and helpful medical tutor. Based on the provided text, generate a JSON object with exactly ONE question (as \'instruction\') and ONE answer (as \'output\').\n\n- The content must relate to health, exercise, sports, fitness, or physiotherapy.\n- Do not include multiple questions or answers.\n- Do not repeat the instruction in the output.\n- Keep the output brief and informative.\n- If the text is not relevant, return: {"instruction": "NULL", "output": "NULL"}\n\n- Respond ONLY with the JSON object. Do NOT include any explanation or commentary.</s>\n<|user|>\nThe Lifter\'s Guide to Fixing Common Injuries and Building a Strong Foundation for Enhancing Performance Dr. Aaron Horschig with Dr. Kevin Sonthana The Lifter\'s Guide to Fixing Common Injuries and Building a Strong Foundation for Enhancing Performance Dr. Aaron Horschig with Dr. Kevin Sonthana Victory Belt Publishing Inc. Las Vegas First published in 2021 by Victory Belt Publishing Inc. Copyright © 2

In [7]:
pprint(raw)

['<|system|>\n'
 'You are a concise and helpful medical tutor. Based on the provided text, '
 "generate a JSON object with exactly ONE question (as 'instruction') and ONE "
 "answer (as 'output').\n"
 '\n'
 '- The content must relate to health, exercise, sports, fitness, or '
 'physiotherapy.\n'
 '- Do not include multiple questions or answers.\n'
 '- Do not repeat the instruction in the output.\n'
 '- Keep the output brief and informative.\n'
 '- If the text is not relevant, return: {"instruction": "NULL", "output": '
 '"NULL"}\n'
 '\n'
 '- Respond ONLY with the JSON object. Do NOT include any explanation or '
 'commentary.</s>\n'
 '<|user|>\n'
 "The Lifter's Guide to Fixing Common Injuries and Building a Strong "
 'Foundation for Enhancing Performance Dr. Aaron Horschig with Dr. Kevin '
 "Sonthana The Lifter's Guide to Fixing Common Injuries and Building a Strong "
 'Foundation for Enhancing Performance Dr. Aaron Horschig with Dr. Kevin '
 'Sonthana Victory Belt Publishing Inc. Las V

In [12]:
from generated_prompt import prompt_template


In [19]:
def generate_content(n_chunks_intervals=None, n_repetitions=3, json_output_name=None, batch_size=16):
    text_chunks = gc.get_text()
    raw_outputs = []
    samples = text_chunks[:] if n_chunks_intervals == None else text_chunks[n_chunks_intervals[0]:n_chunks_intervals[1]]

    qa_gen = gc.get_transformers_pipeline()
    for i in tqdm(range(0, len(samples), batch_size)):
        print(f"i: {i}")
        batch = samples[i:i + batch_size]
        print(f"Processing batch with samples {i, i + batch_size} ")

        for _ in range(n_repetitions):  # Repeat generation 3 times per batch
            prompt = [prompt_template(chunk,1) for chunk in batch]
            #prompt = [gc.get_prompt(chunk) for chunk in batch]
            
            print(prompt)
            raw_output = qa_gen(
                prompt, 
                max_new_tokens=256, 
                do_sample=True,
                temperature=0.7,
                top_k=50,
                top_p=0.95
            )
            
            raw_outputs.extend([o[0]["generated_text"] for o in raw_output])
    
    if json_output_name:
        #with open(f"raw_outputs_samples{len(samples)}_nreps{n_repetitions}_new_tok{256}_chuck_size{250}_overlap{30}.json", "w") as f:
        with open(f"{json_output_name}.json", "w") as f:
            json.dump(raw_outputs, f)
    
    return raw_outputs, raw_output


In [20]:
raw_outputs, raw_output = generate_content(n_chunks_intervals=[35,36], n_repetitions=1, json_output_name=None, batch_size=1)


Device set to use cuda:0
  0%|          | 0/1 [00:00<?, ?it/s]

i: 0
Processing batch with samples (0, 1) 
['You are an expert data curator assisting a machine learning engineer in creating a high-quality instruction tuning dataset. Your task is to transform \n    the provided data chunk into diverse question and answer (Q&A) pairs that will be used to fine-tune a language model. \n\n    For each of the 1 entries, generate one or two well-structured questions that reflect different aspects of the information in the chunk. \n    Ensure a mix of longer and shorter questions, with shorter ones typically containing 1-2 sentences and longer ones spanning up to 3-4 sentences. Each \n    Q&A pair should be concise yet informative, capturing key insights from the data.\n\n    Structure your output in JSON format, where each object contains \'question\' and \'answer\' fields. The JSON structure should look like this:\n\n        "question": "Your question here...",\n        "answer": "Your answer here..."\n\n    Focus on creating clear, relevant, and varied 

100%|██████████| 1/1 [00:10<00:00, 10.88s/it]


In [21]:
pprint(raw_outputs)

['You are an expert data curator assisting a machine learning engineer in '
 'creating a high-quality instruction tuning dataset. Your task is to '
 'transform \n'
 '    the provided data chunk into diverse question and answer (Q&A) pairs '
 'that will be used to fine-tune a language model. \n'
 '\n'
 '    For each of the 1 entries, generate one or two well-structured questions '
 'that reflect different aspects of the information in the chunk. \n'
 '    Ensure a mix of longer and shorter questions, with shorter ones '
 'typically containing 1-2 sentences and longer ones spanning up to 3-4 '
 'sentences. Each \n'
 '    Q&A pair should be concise yet informative, capturing key insights from '
 'the data.\n'
 '\n'
 '    Structure your output in JSON format, where each object contains '
 "'question' and 'answer' fields. The JSON structure should look like this:\n"
 '\n'
 '        "question": "Your question here...",\n'
 '        "answer": "Your answer here..."\n'
 '\n'
 '    Focus on crea

In [22]:
model_id = "meta-llama/Llama-3.1-8B-Instruct" #TinyLlama/TinyLlama-1.1B-Chat-v1.0"
gc = GenerateQAContent(file_name, model_id)

In [29]:
def generate_content(n_chunks_intervals=None, n_repetitions=3, json_output_name=None, batch_size=16):
    text_chunks = gc.get_text()
    raw_outputs = []
    samples = text_chunks[:] if n_chunks_intervals == None else text_chunks[n_chunks_intervals[0]:n_chunks_intervals[1]]

    qa_gen = gc.get_transformers_pipeline()
    for i in tqdm(range(0, len(samples), batch_size)):
        print(f"i: {i}")
        batch = samples[i:i + batch_size]
        print(f"Processing batch with samples {i, i + batch_size} ")

        for _ in range(n_repetitions):  # Repeat generation 3 times per batch
            prompt = [prompt_template(chunk, 1) for chunk in batch]
            #prompt = [gc.get_prompt(chunk) for chunk in batch]
            
            print(prompt)
            raw_output = qa_gen(
                prompt, 
                max_new_tokens=256, 
                do_sample=True,
                temperature=0.7,
                top_k=50,
                top_p=0.95
            )
            
            raw_outputs.extend([o[0]["generated_text"] for o in raw_output])
    
    if json_output_name:
        #with open(f"raw_outputs_samples{len(samples)}_nreps{n_repetitions}_new_tok{256}_chuck_size{250}_overlap{30}.json", "w") as f:
        with open(f"{json_output_name}.json", "w") as f:
            json.dump(raw_outputs, f)
    
    return raw_outputs, raw_output

In [30]:
raw_outputs, raw_output = generate_content(n_chunks_intervals=[35,36], n_repetitions=1, json_output_name=None, batch_size=1)


ValueError: Some modules are dispatched on the CPU or the disk. Make sure you have enough GPU RAM to fit the quantized model. If you want to dispatch the model on the CPU or the disk while keeping these modules in 32-bit, you need to set `llm_int8_enable_fp32_cpu_offload=True` and pass a custom `device_map` to `from_pretrained`. Check https://huggingface.co/docs/transformers/main/en/main_classes/quantization#offload-between-cpu-and-gpu for more details. 

['You are an expert data curator assisting a machine learning engineer in creating a high-quality instruction tuning dataset. Your task is to transform \n    the provided data chunk into diverse question and answer (Q&A) pairs that will be used to fine-tune a language model. \n\n    For each of the 5 entries, generate one or two well-structured questions that reflect different aspects of the information in the chunk. \n    Ensure a mix of longer and shorter questions, with shorter ones typically containing 1-2 sentences and longer ones spanning up to 3-4 sentences. Each \n    Q&A pair should be concise yet informative, capturing key insights from the data.\n\n    Structure your output in JSON format, where each object contains \'question\' and \'answer\' fields. The JSON structure should look like this:\n\n        "question": "Your question here...",\n        "answer": "Your answer here..."\n\n    Focus on creating clear, relevant, and varied questions that encourage the model to learn from diverse perspectives. Avoid any sensitive or biased \n    content, ensuring answers are accurate and neutral.\n\n    Example:\n    \n        "question": "What is the primary purpose of this dataset?",\n        "answer": "This dataset serves as training data for fine-tuning a language model."\n    \n\n    By following these guidelines, you\'ll contribute to a robust and effective dataset that enhances the model\'s performance."\n\n    ---\n\n    **Explanation:**\n\n    - **Clarity and Specificity:** The revised prompt clearly defines the role of the assistant and the importance of the task, ensuring alignment with the \n    project goals.\n    - **Quality Standards:** It emphasizes the need for well-formulated Q&A pairs, specifying the structure and content of each question and answer.\n    - **Output Format:** An example JSON structure is provided to guide the format accurately.\n    - **Constraints and Biases:** A note on avoiding sensitive or biased content ensures ethical considerations are met.\n    - **Step-by-Step Guidance:** The prompt breaks down the task into manageable steps, making it easier for the assistant to follow.\n\n    This approach ensures that the generated data is both high-quality and meets the specific requirements of the machine learning project.\n    \n    Data\n    Does this mean every rounded-spine lift will create a bulging disc? Not necessarily. Many factors play into this discussion. Every athlete tolerates the forces of bending their spine differently. This is why elite gymnasts can bend themselves in half over and over, yet attempting the same movements would eventually spell disaster for an elite heavyweight powerlifter. Here\'s a great way to understand this principle. Some tree branches are slender and bend easily over and over again. Other branches are thicker and begin to snap in two after a few bends. Every body is different. Depending on a number of factors, such as your anatomy, genetics, the amount of weight lifted, and the degree of poor technique, your body may be more or less resilient to developing a disc bulge. 18 This doesn\'t mean you should fear exion of the spine. However, you must understand that the mechanism that creates a disc bulge includes exion. If the force applied to the spine as it exes is low, power generation remains low, and so does injury risk. This is why the cat-camel exercise (moving the spine through a full range in and out of exion but under low load) is a great option for many people. It is not until we introduce force into the equation that things begin to change. Flexion by itself isn\'t the problem.\n\n     "name": "Disc Bulge Injury Prevention"\n     "description": "Learn how to prevent disc bulge injuries through proper understanding of spinal exion, anatomy, and training techniques."\n     "category": "fitness"\n     "subcategory": "injury prevention"\n     "keywords": "disc bulge, injury prevention, fitness"\n     "url": "https://example.com/disc-bulge-injury-prevention"\n\n    Here is the data in a JSON format:\n\n    {\n        "name": "Disc Bulge Injury Prevention",\n        "description": "Learn how to prevent disc bulge injuries through proper understanding of spinal exion, anatomy, and training techniques.",\n        "category": "fitness",\n        "subcategory": "injury prevention",\n        "keywords": "disc bulge, injury prevention, fitness",\n        "url": "https://example.com/disc-bulge-injury-prevention"\n    }\n\n    Q&A Pairs:\n    1. Question: What is the primary purpose of this dataset?\n        Answer: This dataset serves as training data for fine-tuning a language model.\n\n    2. Question: What factors contribute to the development of a disc bulge?\n        Answer: Factors such as anatomy, genetics, weight lifted, and poor', 'You are an expert data curator assisting a machine learning engineer in creating a high-quality instruction tuning dataset. Your task is to transform \n    the provided data chunk into diverse question and answer (Q&A) pairs that will be used to fine-tune a language model. \n\n    For each of the 5 entries, generate one or two well-structured questions that reflect different aspects of the information in the chunk. \n    Ensure a mix of longer and shorter questions, with shorter ones typically containing 1-2 sentences and longer ones spanning up to 3-4 sentences. Each \n    Q&A pair should be concise yet informative, capturing key insights from the data.\n\n    Structure your output in JSON format, where each object contains \'question\' and \'answer\' fields. The JSON structure should look like this:\n\n        "question": "Your question here...",\n        "answer": "Your answer here..."\n\n    Focus on creating clear, relevant, and varied questions that encourage the model to learn from diverse perspectives. Avoid any sensitive or biased \n    content, ensuring answers are accurate and neutral.\n\n    Example:\n    \n        "question": "What is the primary purpose of this dataset?",\n        "answer": "This dataset serves as training data for fine-tuning a language model."\n    \n\n    By following these guidelines, you\'ll contribute to a robust and effective dataset that enhances the model\'s performance."\n\n    ---\n\n    **Explanation:**\n\n    - **Clarity and Specificity:** The revised prompt clearly defines the role of the assistant and the importance of the task, ensuring alignment with the \n    project goals.\n    - **Quality Standards:** It emphasizes the need for well-formulated Q&A pairs, specifying the structure and content of each question and answer.\n    - **Output Format:** An example JSON structure is provided to guide the format accurately.\n    - **Constraints and Biases:** A note on avoiding sensitive or biased content ensures ethical considerations are met.\n    - **Step-by-Step Guidance:** The prompt breaks down the task into manageable steps, making it easier for the assistant to follow.\n\n    This approach ensures that the generated data is both high-quality and meets the specific requirements of the machine learning project.\n    \n    Data\n    Does this mean every rounded-spine lift will create a bulging disc? Not necessarily. Many factors play into this discussion. Every athlete tolerates the forces of bending their spine differently. This is why elite gymnasts can bend themselves in half over and over, yet attempting the same movements would eventually spell disaster for an elite heavyweight powerlifter. Here\'s a great way to understand this principle. Some tree branches are slender and bend easily over and over again. Other branches are thicker and begin to snap in two after a few bends. Every body is different. Depending on a number of factors, such as your anatomy, genetics, the amount of weight lifted, and the degree of poor technique, your body may be more or less resilient to developing a disc bulge. 18 This doesn\'t mean you should fear exion of the spine. However, you must understand that the mechanism that creates a disc bulge includes exion. If the force applied to the spine as it exes is low, power generation remains low, and so does injury risk. This is why the cat-camel exercise (moving the spine through a full range in and out of exion but under low load) is a great option for many people. It is not until we introduce force into the equation that things begin to change. Flexion by itself isn\'t the problem.\n\n     Here are the 5 entries of the provided data chunk:\n\n    1. Does this mean every rounded-spine lift will create a bulging disc? Not necessarily. Many factors play into this discussion. Every athlete tolerates the forces of bending their spine differently. This is why elite gymnasts can bend themselves in half over and over, yet attempting the same movements would eventually spell disaster for an elite heavyweight powerlifter. Here\'s a great way to understand this principle. Some tree branches are slender and bend easily over and over again. Other branches are thicker and begin to snap in two after a few bends. Every body is different. Depending on a number of factors, such as your anatomy, genetics, the amount of weight lifted, and the degree of poor technique, your body may be more or less resilient to developing a disc bulge.\n    2. This doesn\'t mean you should fear exion of the spine. However, you must understand that the mechanism that creates a disc bulge includes exion. If the force applied to the spine as it exes is low, power generation remains low, and so does injury risk. This is why the cat-camel exercise (moving the spine through a full range in and out of exion but under low load']

{"instruction": "What is the primary cause of a disc bulge?", "output": "The primary cause of a disc bulge is the combination of flexion and the force applied to the spine, rather than flexion alone."

In [ ]:
pprint(raw_outputs)

['<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n'
 '\n'
 'Cutting Knowledge Date: December 2023\n'
 'Today Date: 26 Jul 2024\n'
 '\n'
 'You are a concise and helpful medical tutor. Based on the provided text, '
 "generate a JSON object with exactly ONE question (as 'instruction') and ONE "
 "answer (as 'output').\n"
 '\n'
 '- The content must relate to health, exercise, sports, fitness, or '
 'physiotherapy.\n'
 '- Do not include multiple questions or answers.\n'
 '- Do not repeat the instruction in the output.\n'
 '- Keep the output brief and informative.\n'
 '- If the text is not relevant, return: {"instruction": "NULL", "output": '
 '"NULL"}\n'
 '\n'
 '- Respond ONLY with the JSON object. Do NOT include any explanation or '
 'commentary.<|eot_id|><|start_header_id|>user<|end_header_id|>\n'
 '\n'
 'Does this mean every rounded-spine lift will create a bulging disc? Not '
 'necessarily. Many factors play into this discussion. Every athlete tolerates '
 'the forces of be

In [26]:
pprint(raw_outputs)

['<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n'
 '\n'
 'Cutting Knowledge Date: December 2023\n'
 'Today Date: 26 Jul 2024\n'
 '\n'
 'You are a concise and helpful medical tutor. Based on the provided text, '
 "generate a JSON object with exactly ONE question (as 'instruction') and ONE "
 "answer (as 'output').\n"
 '\n'
 '- The content must relate to health, exercise, sports, fitness, or '
 'physiotherapy.\n'
 '- Do not include multiple questions or answers.\n'
 '- Do not repeat the instruction in the output.\n'
 '- Keep the output brief and informative.\n'
 '- If the text is not relevant, return: {"instruction": "NULL", "output": '
 '"NULL"}\n'
 '\n'
 '- Respond ONLY with the JSON object. Do NOT include any explanation or '
 'commentary.<|eot_id|><|start_header_id|>user<|end_header_id|>\n'
 '\n'
 'Does this mean every rounded-spine lift will create a bulging disc? Not '
 'necessarily. Many factors play into this discussion. Every athlete tolerates '
 'the forces of be

## Show converted document

In [3]:
show_document_test=True
if show_document_test:
    n = 10
    for i in range(n):
        pprint(result.document.export_to_dict()['texts'][i]['text'])
        print('**' * 20)


'REBUILDING MILO'
****************************************
("The Lifter's Guide to Fixing Common Injuries and Building a Strong "
 'Foundation for Enhancing Performance')
****************************************
'Dr. Aaron Horschig with Dr. Kevin Sonthana'
****************************************
'REBUILDING MILO'
****************************************
("The Lifter's Guide to Fixing Common Injuries and Building a Strong "
 'Foundation for Enhancing Performance')
****************************************
'Dr. Aaron Horschig with Dr. Kevin Sonthana'
****************************************
'Victory Belt Publishing Inc. Las Vegas'
****************************************
'First published in 2021 by Victory Belt Publishing Inc.'
****************************************
'Copyright © 2021 Aaron Horschig and Dr. Kevin Sonthana'
****************************************
'All rights reserved'
****************************************


In [8]:
result[0]

TypeError: 'ConversionResult' object is not subscriptable

In [9]:
#for i, item in enumerate(result.document.texts):
for i in range(90,140):
    print(f"--- Text {i} ---")
    print(repr(result.document.texts[i].text))  # shows \n, \t, etc.


--- Text 90 ---
"Seriously? They knew I had a huge competition coming up soon. I couldn't just stop lifting."
--- Text 91 ---
"So I resorted to doing what most athletes do in this situation. Before each training session, I would pop three or four Advil and slather Icy Hot all over my knee, and I iced my knee at the end of each day. While I was able to make it through the rest of my training cycle without further injury, I was frustrated by how the competition went. I placed sixth in my weight class, but I knew I could have performed better if my training hadn't been interrupted by this injury."
--- Text 92 ---
"I tell you these stories to remind you that all strength athletes deal with pain. While Josiah's life-threatening injury is the exception rather than the rule, the underlying theme of both stories remains true. When you train hard and don't give your body enough time to recover, the stresses of training begin to accumulate. And when you mix in less-thanperfect lifting technique,

In [11]:
for i, item in enumerate(result.document.texts[90:140], start=100):
    visible = item.text.replace("\n", "\\n").replace("\t", "\\t")
    print(f"--- Text {i} ---\n{visible}")

--- Text 100 ---
Seriously? They knew I had a huge competition coming up soon. I couldn't just stop lifting.
--- Text 101 ---
So I resorted to doing what most athletes do in this situation. Before each training session, I would pop three or four Advil and slather Icy Hot all over my knee, and I iced my knee at the end of each day. While I was able to make it through the rest of my training cycle without further injury, I was frustrated by how the competition went. I placed sixth in my weight class, but I knew I could have performed better if my training hadn't been interrupted by this injury.
--- Text 102 ---
I tell you these stories to remind you that all strength athletes deal with pain. While Josiah's life-threatening injury is the exception rather than the rule, the underlying theme of both stories remains true. When you train hard and don't give your body enough time to recover, the stresses of training begin to accumulate. And when you mix in less-thanperfect lifting technique, t

In [ ]:
from docling.document_converter import DocumentConverter, DocumentConverterOptions
from docling.document_converter import DocumentConverter


# Configure options to preserve line breaks
options = DocumentConverterOptions(
    preserve_line_breaks=True,    # Keep original \n from the PDF text stream
    merge_paragraphs=False,       # Don't merge multiple lines into a single paragraph
    normalize_whitespace=False,   # Keep spacing as is
    extract_images=False          # Optional: skip image extraction for speed
)

converter = DocumentConverter(options=options)
file_short="../data/books/rebuilding_milo_p90to140.pdf"

result = converter.convert(file_short)

# Now you should see \n in the text output
for i, item in enumerate(result.document.texts[:]):
    print(f"--- Text {i} ---")
    print(repr(item.text))  # repr() makes hidden chars like \n visible

ImportError: cannot import name 'DocumentConverterOptions' from 'docling.document_converter' (/home/mrosaria/Projects/NLP/GymRat/ratenv/lib/python3.12/site-packages/docling/document_converter.py)